# Huffman Coding Helper Functions: 

In [1]:
import struct
import heapq
from collections import defaultdict, Counter
from PIL import Image

# Function to build Huffman Tree
class Node:
    def __init__(self, freq, symbol=None, left=None, right=None):
        self.freq = freq
        self.symbol = symbol
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(frequencies):
    heap = [Node(freq, symbol) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)
    
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = Node(left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)
    
    return heap[0]

# Function to generate Huffman codes from the tree
def generate_codes(node, prefix="", codebook={}):
    if node.symbol is not None:
        codebook[node.symbol] = prefix
    else:
        generate_codes(node.left, prefix + "0", codebook)
        generate_codes(node.right, prefix + "1", codebook)
    return codebook

# Function to encode data using the Huffman codes
def huffman_encode(data, codes):
    encoded_data = ''.join([codes[byte] for byte in data])
    return encoded_data

# Function to apply Huffman encoding to a color channel
def encode_channel(channel_data):
    frequency = Counter(channel_data)
    #print(frequency)
    huffman_tree = build_huffman_tree(frequency)
    codes = generate_codes(huffman_tree)
    encoded_data = huffman_encode(channel_data, codes)
    return encoded_data, codes



# Various Read Functions 

In [2]:
from PIL import Image

# Function to read PNG file and extract pixel data
def read_png(file_path):
    # Open the image using Pillow
    img = Image.open(file_path)
    
    # Convert image to RGB (just in case it's not in RGB mode)
    img = img.convert('RGB')
    
    # Get the width and height of the image
    width, height = img.size
    
    # Extract pixel data (R, G, B values) as a list of tuples
    pixel_data = []
    for y in range(height):
        for x in range(width):
            r, g, b = img.getpixel((x, y))
            pixel_data.append((r, g, b))
    #print(pixel_data[:100])
    return pixel_data

def combine_pairs(channel_data):
        combined_data = []
        for i in range(0, len(channel_data) - 1, 2):
            # Combine two values into a single bitstring
            combined_value = (channel_data[i] << 6) | channel_data[i + 1]
            combined_data.append(combined_value)
        # Handle case if there's an odd number of values by appending the last one as is
        if len(channel_data) % 2 != 0:
            combined_data.append(channel_data[-1])
        return combined_data
    
# Function to read BMP file and extract pixel data
def read_bmp(file_path):
    with open(file_path, 'rb') as f:
        # Read header (first 54 bytes)
        header = f.read(54)
        
        # Extract width, height, and pixel data offset from the header
        width, height = struct.unpack('<ii', header[18:26])
        pixel_data_offset = struct.unpack('<I', header[10:14])[0]
        
        # Move to the pixel data offset
        f.seek(pixel_data_offset)
        
        # Read pixel data
        pixel_data = []
        row_size = (width * 3 + 3) & ~3  # Row size is padded to the nearest multiple of 4
        for _ in range(height):
            row = f.read(row_size)
            for i in range(0, width * 3, 3):
                b = row[i]
                g = row[i + 1]
                r = row[i + 2]
                pixel_data.append((r, g, b))
        
        return pixel_data
    
def read_dataset_file(image_data):
    width = 32 
    height = 32
    pixel_data = image_data.reshape(-1,3)
    print(pixel_data.shape)
    return width, height, pixel_data 

# Main function to perform Huffman encoding on BMP file
def huffman_encoding_png(file_path):
    # Step 1: Read PNG file and extract pixel data
    pixel_data = read_png(file_path)
    
    # Step 2: Separate the R, G, B channels
    reds = [r>>2 for r, g, b in pixel_data]
    greens = [g>>2 for r, g, b in pixel_data]
    blues = [b>>2 for r, g, b in pixel_data]
    
    combined_reds = combine_pairs(reds)
    combined_greens = combine_pairs(greens)
    combined_blues = combine_pairs(blues)

    # Step 4: Perform Huffman encoding on each combined channel
    red_encoded, red_codes = encode_channel(combined_reds)
    green_encoded, green_codes = encode_channel(combined_greens)
    blue_encoded, blue_codes = encode_channel(combined_blues)
    
    # Return the encoded data and codes for each channel
    return {
        'red_encoded': red_encoded,
        'red_codes': red_codes,
        'green_encoded': green_encoded,
        'green_codes': green_codes,
        'blue_encoded': blue_encoded,
        'blue_codes': blue_codes
    }
# Function to calculate storage comparison
def calculate_storage_comparison(encoded_data):
    # Step 1: Retrieve the image's width and height from encoded_data
    width = 512
    height = 512
    
    # Step 2: Calculate the length of each encoded channel
    red_encoded_len = len(encoded_data['red_encoded'])
    green_encoded_len = len(encoded_data['green_encoded'])
    blue_encoded_len = len(encoded_data['blue_encoded'])
    
    # Print the lengths of the encoded data for each channel
    print(f"Length of red_encoded: {red_encoded_len} bits")
    print(f"Length of green_encoded: {green_encoded_len} bits")
    print(f"Length of blue_encoded: {blue_encoded_len} bits")
    
    # Step 3: Calculate total storage of the encoded data (sum of all three channels)
    total_encoded_storage = red_encoded_len + green_encoded_len + blue_encoded_len
    print(f"Total storage for Huffman encoded data: {total_encoded_storage} bits")
    
    # Step 4: Calculate storage for the base image (width * height * 24 bits per pixel)
    base_image_storage = width * height * 24  # Width * Height pixels, 24 bits per pixel
    print(f"Base image storage (uncompressed): {base_image_storage} bits")
    
    # Step 5: Compare the two
    compression_ratio = base_image_storage / total_encoded_storage
    print(f"Compression Ratio: {compression_ratio:.2f}x")
    
    return {
        "width": width,
        "height": height,
        "red_encoded_len": red_encoded_len,
        "green_encoded_len": green_encoded_len,
        "blue_encoded_len": blue_encoded_len,
        "total_encoded_storage": total_encoded_storage,
        "base_image_storage": base_image_storage,
        "compression_ratio": compression_ratio
    }

file_path = '/kaggle/input/blackbuckpng/blackbuck.png'  # Replace with your BMP file path
encoded_data = huffman_encoding_png(file_path)
comparison_result = calculate_storage_comparison(encoded_data)


Length of red_encoded: 502928 bits
Length of green_encoded: 528038 bits
Length of blue_encoded: 499387 bits
Total storage for Huffman encoded data: 1530353 bits
Base image storage (uncompressed): 6291456 bits
Compression Ratio: 4.11x


# Results on CIFAR-10

In [4]:
import numpy as np
from PIL import Image
import keras

# Function to handle image arrays like CIFAR-10
def read_dataset_file(image_data):
    # Extract width, height, and channel information
    width, height, channels = image_data.shape
    
    # Flatten the image data to (width * height, 3)
    pixel_data = image_data.reshape(-1, channels)
    return width, height, pixel_data

def combine_pairs(channel_data):
        combined_data = []
        for i in range(0, len(channel_data) - 1, 2):
            # Combine two values into a single bitstring
            combined_value = (channel_data[i] << 6) | channel_data[i + 1]
            combined_data.append(combined_value)
        # Handle case if there's an odd number of values by appending the last one as is
        if len(channel_data) % 2 != 0:
            combined_data.append(channel_data[-1])
        return combined_data

# Function to simulate Huffman encoding (currently just separating RGB)
def huffman_encoding_datasetfile(image_data):
    # Read and extract pixel data from the image
    width, height, pixel_data = read_dataset_file(image_data)
    
    # Separate the R, G, B channels
    reds = pixel_data[:, 0]
    greens = pixel_data[:, 1]
    blues = pixel_data[:, 2]
    
    combined_reds = combine_pairs(reds)
    combined_greens = combine_pairs(greens)
    combined_blues = combine_pairs(blues)

    # Step 4: Perform Huffman encoding on each combined channel
    red_encoded, red_codes = encode_channel(combined_reds)
    green_encoded, green_codes = encode_channel(combined_greens)
    blue_encoded, blue_codes = encode_channel(combined_blues)
    
    # Total encoded size
    total_encoded_size = len(red_encoded) + len(green_encoded) + len(blue_encoded)
    
    # Print sizes for comparison
#    print(f"Original Image Size: {width * height * 3} bytes")
#     print(f"Encoded Red Channel Size: {len(red_encoded)} bytes")
#     print(f"Encoded Green Channel Size: {len(green_encoded)} bytes")
#     print(f"Encoded Blue Channel Size: {len(blue_encoded)} bytes")
#   print(f"Total Encoded Data Size: {total_encoded_size} bytes")
    base_image_storage = 32*32*24
    
    compression_ratio = base_image_storage / total_encoded_size
    #print(f"Compression Ratio: {compression_ratio:.2f}x")
    return compression_ratio

# Example usage with CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(f"Image Shape: {x_train[0].shape}")  # Should print (32, 32, 3)

# Encode the first image in the dataset and compare sizes
random_indices = np.random.choice(len(x_train), 100, replace=False)
total_compression = 0
for i in random_indices:
    compression_ratio = huffman_encoding_datasetfile(x_train[i])
    total_compression += compression_ratio
print(f"Average Compression is: {total_compression/100}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step
Image Shape: (32, 32, 3)
Average Compression is: 1.900140333314159


# Results on CIFAR-100

In [5]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data()
print(f"Image Shape: {x_train[0].shape}")  # Should print (32, 32, 3)

random_indices = np.random.choice(len(x_train), 100, replace=False)
total_compression = 0
for i in random_indices:
    compression_ratio = huffman_encoding_datasetfile(x_train[i])
    total_compression += compression_ratio
print(f"Average Compression is: {total_compression/100}")

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step
Image Shape: (32, 32, 3)
Average Compression is: 1.9200398990420744


# Image.net Dataset

In [6]:
from datasets import load_dataset

from PIL import Image
import numpy as np

def read_dataset_file(image_data):
    # Extract width and height of the image
    width, height = image_data.size
    
    # Convert image to numpy array and reshape to extract pixel data
    # Pixel data will be of shape (width * height, 3) where 3 represents the R, G, B channels
    pixel_data = np.array(image_data).reshape((width * height, 3))
    return width, height, pixel_data

def combine_pairs(channel_data):
        combined_data = []
        for i in range(0, len(channel_data) - 1, 2):
            # Combine two values into a single bitstring
            combined_value = (channel_data[i] << 8) | channel_data[i + 1]
            combined_data.append(combined_value)
        # Handle case if there's an odd number of values by appending the last one as is
        if len(channel_data) % 2 != 0:
            combined_data.append(channel_data[-1])
        return combined_data

def huffman_encoding_datasetfile(image_data):
    # Read and extract pixel data from the image
    width, height, pixel_data = read_dataset_file(image_data)
    
    # Separate the R, G, B channels
    reds = pixel_data[:, 0]
    greens = pixel_data[:, 1]
    blues = pixel_data[:, 2]
    
    combined_reds = combine_pairs(reds>>2)
    combined_greens = combine_pairs(greens>>2)
    combined_blues = combine_pairs(blues>>2)

    # Step 4: Perform Huffman encoding on each combined channel
    red_encoded, red_codes = encode_channel(combined_reds)
    green_encoded, green_codes = encode_channel(combined_greens)
    blue_encoded, blue_codes = encode_channel(combined_blues)
    
    # Total encoded size
    total_encoded_size = len(red_encoded) + len(green_encoded) + len(blue_encoded)
    
    # Print sizes for comparison
#    print(f"Original Image Size: {width * height * 3} bytes")
#     print(f"Encoded Red Channel Size: {len(red_encoded)} bytes")
#     print(f"Encoded Green Channel Size: {len(green_encoded)} bytes")
#     print(f"Encoded Blue Channel Size: {len(blue_encoded)} bytes")
#    print(f"Total Encoded Data Size: {total_encoded_size} bytes")
    base_image_storage = 64*64*24
    
    compression_ratio = base_image_storage / total_encoded_size
    #print(f"Compression Ratio: {compression_ratio:.2f}x")
    return compression_ratio

ds = load_dataset('Maysee/tiny-imagenet', split='train')

total_compression = 0
for i in range(0,50):
    image = ds[i]['image']
    compression_ratio = huffman_encoding_datasetfile(image)
    total_compression += compression_ratio
print(f"Average Compression is: {total_compression/50}")

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

(…)-00000-of-00001-1359597a978bc4fa.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

(…)-00000-of-00001-70d52db3c749a935.parquet:   0%|          | 0.00/14.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Average Compression is: 2.0447827775430905
